# Synthetic HDR volume ratio

Compute the same `hdr_volume_ratio` added by `SyntheticExperimentRunner`, using every saved run under `benchmark/results/synthetic`. No models are retrained and the saved result files are not modified.

The metric is

`analytic HDR volume / volume of the learned transport image of the matching Gaussian ball`.

A ratio near 1 means that the learned region has approximately the same volume as the analytic HDR.

In [1]:
from __future__ import annotations

import gc
import sys
from pathlib import Path

import pandas as pd
import torch


def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")


REPOSITORY_ROOT = find_repository_root(Path.cwd())
RESULTS_ROOT = REPOSITORY_ROOT / "benchmark/results/synthetic"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

sys.path.insert(0, str(REPOSITORY_ROOT))
sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

print(f"Repository: {REPOSITORY_ROOT}")
print(f"Results:    {RESULTS_ROOT}")
print(f"Device:     {DEVICE}")

Repository: /Users/vladimir.kondratyev/minimal_volume_conformal_prediction
Results:    /Users/vladimir.kondratyev/minimal_volume_conformal_prediction/benchmark/results/synthetic
Device:     cpu


In [2]:
from configs.predictors.transport import (
    ConvexPotentialFlowPredictorConfig,
    FlowMatchingPredictorConfig,
    NeuralOptimalTransportPredictorConfig,
    NeuralSplineFlowPredictorConfig,
    NormalizingFlowPredictorConfig,
)
from data.datasets import synthetic as synthetic_datasets
from experimentation import (
    compute_hdr_volume_ratio,
    load_experiment_config,
    validate_synthetic_experiment_config,
)
from predictors import rearranged_transport as rearranged_predictors
from predictors import transport as transport_predictors


PREDICTOR_CONFIG_CLASSES = {
    "convex_potential_flow": ConvexPotentialFlowPredictorConfig,
    "flow_matching": FlowMatchingPredictorConfig,
    "neural_optimal_transport": NeuralOptimalTransportPredictorConfig,
    "neural_spline_flow": NeuralSplineFlowPredictorConfig,
    "normalizing_flow": NormalizingFlowPredictorConfig,
}

PREDICTOR_CLASSES = {
    "convex_potential_flow": transport_predictors.ConvexPotentialFlowPredictor,
    "flow_matching": transport_predictors.FlowMatchingPredictor,
    "neural_optimal_transport": transport_predictors.NeuralOptimalTransportPredictor,
    "neural_spline_flow": transport_predictors.NeuralSplineFlowPredictor,
    "normalizing_flow": transport_predictors.NormalizingFlowPredictor,
}

REARRANGEMENT_CLASSES = {
    "rearranged_transport": rearranged_predictors.RearrangedTransportPredictor,
    "amortized_rearranged_transport": rearranged_predictors.AmortizedRearrangedTransport,
}

DATASET_CLASSES = {
    "banana": synthetic_datasets.BananaDataset,
    "sinusoidal_transport": synthetic_datasets.SinusoidalTransportDataset,
    "star_shaped_gaussian": synthetic_datasets.StarShapedGaussianDataset,
    "star_shaped": synthetic_datasets.StarShapedGaussianDataset,
}


def load_base_predictor(checkpoint_path: Path, predictor_type: str):
    """Load a base predictor while overriding the saved CUDA device if needed."""
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    config_data = dict(checkpoint["config"])
    config_data["device"] = DEVICE
    predictor_config = PREDICTOR_CONFIG_CLASSES[predictor_type].model_validate(
        config_data
    )
    predictor = PREDICTOR_CLASSES[predictor_type](predictor_config)
    predictor.load_state_dict(checkpoint["state_dict"])
    predictor.eval()
    return predictor


def load_final_predictor(run_directory: Path, config):
    if config.rearrangement_config is None:
        checkpoint_path = run_directory / "base/predictor.pt"
        predictor = load_base_predictor(
            checkpoint_path,
            predictor_type=config.predictor_config.type,
        )
        return predictor, "base", checkpoint_path

    checkpoint_path = run_directory / "rearrangement/predictor.pt"
    rearrangement_class = REARRANGEMENT_CLASSES[config.rearrangement_config.type]
    predictor = rearrangement_class.load(
        str(checkpoint_path),
        map_location=DEVICE,
    )
    predictor.eval()
    return predictor, "rearrangement", checkpoint_path


def sample_evaluation_condition(config):
    dataset_config = config.dataset_config.model_copy(update={"device": DEVICE})
    dataset = DATASET_CLASSES[dataset_config.type](dataset_config)
    return dataset.sample_x(1)


def evaluate_run(config_path: Path) -> dict:
    config = load_experiment_config(config_path)
    validate_synthetic_experiment_config(config)

    run_directory = config_path.parent
    predictor, transport_stage, checkpoint_path = load_final_predictor(
        run_directory, config
    )
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)

    conformal_config = config.conformal_config
    comparison = compute_hdr_volume_ratio(
        predictor=predictor,
        condition=sample_evaluation_condition(config),
        coverage_mass=conformal_config.coverage_mass,
        number_of_samples=conformal_config.volume_mc_samples,
        batch_size=conformal_config.volume_batch_size,
        seed=conformal_config.volume_seed,
    )

    record = {
        "dataset": config.dataset_config.type,
        "model": run_directory.parent.name,
        "seed": config.seed,
        "transport_stage": transport_stage,
        "hdr_volume_ratio": comparison["mean"],
        "hdr_volume": comparison["hdr_volume"],
        "transport_ball_volume": comparison["transport_ball_volume"],
        "coverage_mass": comparison["coverage_mass"],
        "volume_mc_samples": comparison["volume_mc_samples"],
        "run_directory": str(run_directory.relative_to(REPOSITORY_ROOT)),
    }

    del predictor
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return record

In [3]:
config_paths = sorted(RESULTS_ROOT.glob("*/*/seed_*/config.json"))
if not config_paths:
    raise FileNotFoundError(f"No saved synthetic runs found under {RESULTS_ROOT}")

print(f"Found {len(config_paths)} saved runs.\n")
records = []
failures = []

for index, config_path in enumerate(config_paths, start=1):
    run_name = config_path.parent.relative_to(RESULTS_ROOT)
    try:
        record = evaluate_run(config_path)
        records.append(record)
        print(
            f"[{index:02d}/{len(config_paths):02d}] {run_name}: "
            f"HDR / transport volume = {record['hdr_volume_ratio']:.6f}"
        )
    except Exception as error:
        failures.append({"run": str(run_name), "error": repr(error)})
        print(f"[{index:02d}/{len(config_paths):02d}] {run_name}: FAILED: {error}")

results = pd.DataFrame(records).sort_values(
    ["dataset", "model", "seed"]
).reset_index(drop=True)
failures = pd.DataFrame(failures)

if not failures.empty:
    print("\nFailures:")
    print(failures.to_string(index=False))

Found 30 saved runs.

[01/30] banana/transport_neural_ot_l2/seed_00: HDR / transport volume = 0.494253
[02/30] banana/transport_neural_ot_l2/seed_01: HDR / transport volume = 0.443140
[03/30] banana/transport_neural_ot_l2/seed_02: HDR / transport volume = 0.394241
[04/30] banana/transport_neural_ot_l2/seed_03: HDR / transport volume = 0.397075
[05/30] banana/transport_neural_ot_l2/seed_04: HDR / transport volume = 0.413410
[06/30] banana/transport_neural_ot_rearranged_l2/seed_00: HDR / transport volume = 0.966428
[07/30] banana/transport_neural_ot_rearranged_l2/seed_01: HDR / transport volume = 0.953798
[08/30] banana/transport_neural_ot_rearranged_l2/seed_02: HDR / transport volume = 0.940957
[09/30] banana/transport_neural_ot_rearranged_l2/seed_03: HDR / transport volume = 0.986160
[10/30] banana/transport_neural_ot_rearranged_l2/seed_04: HDR / transport volume = 0.918577
[11/30] sinusoidal_transport/transport_neural_ot_l2/seed_00: HDR / transport volume = 0.904902
[12/30] sinusoidal

In [4]:
columns = [
    "dataset",
    "model",
    "seed",
    "transport_stage",
    "hdr_volume_ratio",
    "hdr_volume",
    "transport_ball_volume",
    "coverage_mass",
    "volume_mc_samples",
]

print("Per-run results:")
print(
    results[columns].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

summary = (
    results.groupby(["dataset", "model", "transport_stage"], as_index=False)
    .agg(
        mean_hdr_volume_ratio=("hdr_volume_ratio", "mean"),
        std_hdr_volume_ratio=("hdr_volume_ratio", "std"),
        mean_transport_ball_volume=("transport_ball_volume", "mean"),
        number_of_runs=("hdr_volume_ratio", "size"),
    )
)

print("\nSummary across seeds:")
print(summary.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

Per-run results:
             dataset                             model  seed transport_stage  hdr_volume_ratio  hdr_volume  transport_ball_volume  coverage_mass  volume_mc_samples
              banana            transport_neural_ot_l2     0            base          0.494253   14.467569              29.271576       0.900000               1000
              banana            transport_neural_ot_l2     1            base          0.443140   14.467569              32.647859       0.900000               1000
              banana            transport_neural_ot_l2     2            base          0.394241   14.467569              36.697279       0.900000               1000
              banana            transport_neural_ot_l2     3            base          0.397075   14.467569              36.435319       0.900000               1000
              banana            transport_neural_ot_l2     4            base          0.413410   14.467569              34.995723       0.900000               1000

In [3]:
import torch

lv = torch.tensor([
    -1.91376,
    -1.95886,
    -1.91788
])

print(lv.mean()), 
print(lv.std())

tensor(-1.9302)
tensor(0.0249)
